# Stage 09 structure-aware redesign on SageMaker

This perfectly clean notebook runs the complete Stage 09 pipeline:
1. Extract payload to a clean directory and prepare the environment
2. Define a stricter edit space
3. Build the structural-surrogate dataset (skipped if baseline absent)
4. Train/configure the surrogate
5. Run structure-aware localized search
6. Prefilter candidates
7. Validate the top panel with the fixed Stage 08 structural validator
8. Build the final Stage 09 comparison report

In [1]:
from pathlib import Path
import zipfile
import shutil

ZIP_PATH = Path("/home/sagemaker-user/sagemaker_master_bundle.zip")
DEST = Path("/home/sagemaker-user/phageforge_clean")

assert ZIP_PATH.exists(), f"Missing zip: {ZIP_PATH}"

# Clean out any previous failed extractions
if DEST.exists():
    shutil.rmtree(DEST, ignore_errors=True)

DEST.mkdir(parents=True, exist_ok=True)

# Extract while normalizing Windows backslashes to POSIX separators
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    for member in zf.infolist():
        raw_name = member.filename
        norm_name = raw_name.replace("\\", "/").lstrip("/")
        if not norm_name:
            continue
        target = DEST / norm_name
        if raw_name.endswith("/") or raw_name.endswith("\\"):
            target.mkdir(parents=True, exist_ok=True)
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        with zf.open(member) as src, open(target, "wb") as dst:
            dst.write(src.read())

print("Extraction complete to:", DEST)

Extraction complete to: /home/sagemaker-user/phageforge_clean


In [2]:
import subprocess, sys
import os

ROOT = Path("/home/sagemaker-user/phageforge_clean")
os.chdir(ROOT)
print("Working directory set to:", Path.cwd())

subprocess.run([sys.executable, "-m", "pip", "install", "-U", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "fair-esm", "biopython"], check=True)
print("Environment ready.")

Working directory set to: /home/sagemaker-user/phageforge_clean


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 88.6 MB/s  0:00:00


  Attempting uninstall: pip
    Found existing installation: pip 26.0.1
    Uninstalling pip-26.0.1:
      Successfully uninstalled pip-26.0.1


Obtaining file:///home/sagemaker-user/phageforge_clean
  Installing build dependencies: started


  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started


...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/2 [biopython]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/2 [biopython]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/2 [biopython]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [phageforge]


Environment ready.


## Configure Paths

In [3]:
# Use the actual flat paths present in the new clean SageMaker extraction
STAGE07_CONTEXT = "stage07/context/stage07_context.base.json"
STRICT_CSV = "rbp_dataset_eskapee_strict.csv"
STAGE07_RANKED = "stage07/multimodal_rank/final_multimodal_ranked_candidates.csv"
PREDICTOR_MODEL = "seed_42/model.joblib"
LABEL_CLASSES = "seed_42/label_classes.json"
BASELINE_STAGE08 = None

# Use results/stage09 so that Stage 10 can easily find it later
OUT_ROOT = "results/stage09"

for path in [STAGE07_CONTEXT, STRICT_CSV, STAGE07_RANKED, PREDICTOR_MODEL, LABEL_CLASSES]:
    p = Path(path)
    print(f"Checking: {p}")
    assert p.exists(), f"Missing required path: {path}"

Path(OUT_ROOT).mkdir(parents=True, exist_ok=True)
print("Output root:", OUT_ROOT)
print("Repo ready.")

Checking: stage07/context/stage07_context.base.json
Checking: rbp_dataset_eskapee_strict.csv
Checking: stage07/multimodal_rank/final_multimodal_ranked_candidates.csv
Checking: seed_42/model.joblib
Checking: seed_42/label_classes.json
Output root: results/stage09
Repo ready.


## Step 1 — Define a stricter edit space

In [4]:
!python scripts/09a_define_edit_space.py \
  --context_json {STAGE07_CONTEXT} \
  --strict_csv {STRICT_CSV} \
  --output_json {OUT_ROOT}/edit_space/stage09_edit_space.json \
  --max_edit_positions 12 \
  --soft_buffer_positions 6 \
  --min_mutations 3 \
  --max_mutations 8

Wrote: results/stage09/edit_space/stage09_edit_space.json
hard_edit_positions: [291, 329, 334, 349, 373, 401, 427, 505, 527, 585, 592, 615]
soft_edit_positions: [425, 522, 402, 400, 338, 391]


## Step 2 — Build and configure the structural surrogate

In [5]:
SURROGATE_MODEL = None

if BASELINE_STAGE08 is not None and Path(BASELINE_STAGE08).exists():
    subprocess.run(
        [
            sys.executable, "scripts/09b_build_structure_surrogate_dataset.py",
            "--context_json", STAGE07_CONTEXT,
            "--ranked_csv", STAGE07_RANKED,
            "--structural_csv", BASELINE_STAGE08,
            "--out_csv", f"{OUT_ROOT}/surrogate/stage09_surrogate_dataset.csv",
            "--out_json", f"{OUT_ROOT}/surrogate/stage09_surrogate_summary.json",
        ], check=True
    )
    subprocess.run(
        [
            sys.executable, "scripts/09c_train_structure_surrogate.py",
            "--dataset_csv", f"{OUT_ROOT}/surrogate/stage09_surrogate_dataset.csv",
            "--summary_json", f"{OUT_ROOT}/surrogate/stage09_surrogate_summary.json",
            "--out_model", f"{OUT_ROOT}/surrogate/stage09_surrogate.joblib",
        ], check=True
    )
    SURROGATE_MODEL = f"{OUT_ROOT}/surrogate/stage09_surrogate.joblib"
    print("Surrogate model:", SURROGATE_MODEL)
else:
    print("[INFO] No Stage 08 baseline summary CSV is available in this upload.")
    print("[INFO] Skipping 09b/09c and continuing with Stage 09 search without a fitted surrogate.")

[INFO] No Stage 08 baseline summary CSV is available in this upload.
[INFO] Skipping 09b/09c and continuing with Stage 09 search without a fitted surrogate.


## Step 3 — Run structure-aware localized search

In [7]:
cmd = [
    sys.executable, "scripts/09d_localized_search.py",
    "--context_json", STAGE07_CONTEXT,
    "--edit_space_json", f"{OUT_ROOT}/edit_space/stage09_edit_space.json",
    "--strict_csv", STRICT_CSV,
    "--predictor_model", PREDICTOR_MODEL,
    "--label_classes_json", LABEL_CLASSES,
    "--out_csv", f"{OUT_ROOT}/search/stage09_search_candidates.csv",
    "--out_json", f"{OUT_ROOT}/search/stage09_search_run.json",
    "--esm_model", "facebook/esm2_t33_650M_UR50D", # <-- This is the corrected line
    "--batch_size", "4",
    "--rounds", "4",
    "--beam_width", "24",
    "--proposals_per_parent", "18",
    "--max_mutations", "8",
]

if SURROGATE_MODEL is not None:
    cmd += ["--surrogate_model", SURROGATE_MODEL]

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

Running: /opt/conda/bin/python scripts/09d_localized_search.py --context_json stage07/context/stage07_context.base.json --edit_space_json results/stage09/edit_space/stage09_edit_space.json --strict_csv rbp_dataset_eskapee_strict.csv --predictor_model seed_42/model.joblib --label_classes_json seed_42/label_classes.json --out_csv results/stage09/search/stage09_search_candidates.csv --out_json results/stage09/search/stage09_search_run.json --esm_model facebook/esm2_t33_650M_UR50D --batch_size 4 --rounds 4 --beam_width 24 --proposals_per_parent 18 --max_mutations 8


/opt/conda/lib/python3.12/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


2026-05-18 17:58:16.706137: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779127096.721906     773 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779127096.727408     773 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779127096.740918     773 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779127096.740950     773 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779127096.740953     773 computation_placer.cc:177] computation placer alr

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Wrote: results/stage09/search/stage09_search_candidates.csv
Wrote: results/stage09/search/stage09_search_run.json
 sample_id  round_index  stage09_score  target_probability  predicted_mean_plddt  predicted_rmsd  sequence_identity  mutation_count
         1            4       0.335713            0.024834             87.581558         1.41459           0.993921               4
         2            4       0.335432            0.022967             87.581558         1.41459           0.993921               4
         3            4       0.335353            0.030751             87.581558         1.41459           0.993921               4
         4            4       0.335212            0.024119             87.581558         1.41459           0.993921               4
         5            4       0.335212            0.029721             87.581558         1.41459           0.993921               4
         6            4       0.335180            0.031667             87.581558         1.414

CompletedProcess(args=['/opt/conda/bin/python', 'scripts/09d_localized_search.py', '--context_json', 'stage07/context/stage07_context.base.json', '--edit_space_json', 'results/stage09/edit_space/stage09_edit_space.json', '--strict_csv', 'rbp_dataset_eskapee_strict.csv', '--predictor_model', 'seed_42/model.joblib', '--label_classes_json', 'seed_42/label_classes.json', '--out_csv', 'results/stage09/search/stage09_search_candidates.csv', '--out_json', 'results/stage09/search/stage09_search_run.json', '--esm_model', 'facebook/esm2_t33_650M_UR50D', '--batch_size', '4', '--rounds', '4', '--beam_width', '24', '--proposals_per_parent', '18', '--max_mutations', '8'], returncode=0)

## Step 4 — Prefilter candidates before expensive structural validation

In [8]:
!python scripts/09e_structural_prefilter.py \
  --search_csv {OUT_ROOT}/search/stage09_search_candidates.csv \
  --search_meta_json {OUT_ROOT}/search/stage09_search_run.json \
  --out_csv {OUT_ROOT}/prefilter/stage09_prefilter_top12.csv \
  --top_k 12 \
  --max_structural_risk 0.55 \
  --min_predicted_plddt 55 \
  --max_predicted_rmsd 4.5 \
  --min_sequence_identity 0.93

2026-05-18 18:01:00.157405: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered


E0000 00:00:1779127260.174087     867 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779127260.179713     867 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779127260.193275     867 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779127260.193305     867 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779127260.193309     867 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779127260.193312     867 computation_placer.cc:177] computation placer already registered. Please check linka

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t12_35M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Wrote: results/stage09/prefilter/stage09_prefilter_top12.csv
source_target_host: Acinetobacter
 prefilter_rank  sample_id  stage09_score  target_probability  predicted_mean_plddt  predicted_rmsd  sequence_identity
              1          1       0.335713            0.024834              87.58156         1.41459           0.993921
              2          2       0.335432            0.022967              87.58156         1.41459           0.993921
              3          3       0.335353            0.030751              87.58156         1.41459           0.993921
              4          5       0.335212            0.029721              87.58156         1.41459           0.993921
              5          4       0.335212            0.024119              87.58156         1.41459           0.993921
              6          6       0.335180            0.031667              87.58156         1.41459           0.993921
              7          7       0.335049            0.037384           

## Step 5 — Validate top 10 and top 3 with the fixed validator

In [9]:
!echo "--- DISK SPACE ---"
!df -h /home/sagemaker-user

!echo ""
!echo "--- GPU VRAM ---"
!nvidia-smi

--- DISK SPACE ---


Filesystem      Size  Used Avail Use% Mounted on
/dev/nvme2n1     90G  3.3G   87G   4% /home/sagemaker-user


--- GPU VRAM ---


Mon May 18 18:01:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.58.03              Driver Version: 595.58.03      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A10G                    On  |   00000000:00:1E.0 Off |                    0 |
|  0%   40C    P0             61W /  300W |       0MiB /  23028MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [10]:
!python scripts/09f_validate_stage09_candidates.py \
  --prefilter_csv {OUT_ROOT}/prefilter/stage09_prefilter_top12.csv \
  --context_json {STAGE07_CONTEXT} \
  --out_dir {OUT_ROOT}/validation_top10 \
  --top_k 10 \
  --device cuda \
  --chunk_size 128 \
  --num_recycles 1 \
  --resume

!python scripts/09f_validate_stage09_candidates.py \
  --prefilter_csv {OUT_ROOT}/prefilter/stage09_prefilter_top12.csv \
  --context_json {STAGE07_CONTEXT} \
  --out_dir {OUT_ROOT}/validation_top3 \
  --top_k 3 \
  --device cuda \
  --chunk_size 128 \
  --num_recycles 1 \
  --resume

Running:
/opt/conda/bin/python /home/sagemaker-user/phageforge_clean/scripts/08a_structural_fasttrack_validation.py --validated_csv results/stage09/validation_top10/stage09_validated_top10.csv --ranked_csv results/stage09/validation_top10/stage09_ranked_for_validation.csv --context_json stage07/context/stage07_context.base.json --out_dir results/stage09/validation_top10 --top_k 10 --device cuda --chunk_size 128 --num_recycles 1 --resume


[INFO] Attempting merge on columns: ['sample_id', 'generation_regime', 'final_multimodal_rank_score', 'mutation_positions']


2026-05-18 18:01:59.458144: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779127319.474003     904 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779127319.479465     904 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779127319.493158     904 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779127319.493193     904 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779127319.493203     904 computation_placer.cc:177] computation placer alr

tokenizer_config.json: 100%|██████████████████| 40.0/40.0 [00:00<00:00, 693kB/s]


vocab.txt: 100%|█████████████████████████████| 72.0/72.0 [00:00<00:00, 1.41MB/s]


...

Running:
/opt/conda/bin/python /home/sagemaker-user/phageforge_clean/scripts/08a_structural_fasttrack_validation.py --validated_csv results/stage09/validation_top3/stage09_validated_top3.csv --ranked_csv results/stage09/validation_top3/stage09_ranked_for_validation.csv --context_json stage07/context/stage07_context.base.json --out_dir results/stage09/validation_top3 --top_k 3 --device cuda --chunk_size 128 --num_recycles 1 --resume


[INFO] Attempting merge on columns: ['sample_id', 'generation_regime', 'final_multimodal_rank_score', 'mutation_positions']


2026-05-18 18:11:00.255365: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779127860.271373    1140 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779127860.276881    1140 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779127860.290274    1140 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779127860.290304    1140 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779127860.290308    1140 computation_placer.cc:177] computation placer alr

Some weights of EsmForProteinFolding were not initialized from the model checkpoint at facebook/esmfold_v1 and are newly initialized: ['esm.contact_head.regression.bias', 'esm.contact_head.regression.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Wrote: results/stage09/validation_top3/stage08_structural_fasttrack_summary.csv
Wrote: results/stage09/validation_top3/stage08_structural_fasttrack_summary.json
Wrote: results/stage09/validation_top3/stage08_structural_fasttrack_report.md
Wrote seed PDB: results/stage09/validation_top3/pdbs/seed_selected_seed.pdb
Wrote candidate PDB: results/stage09/validation_top3/pdbs/candidate_1.pdb
Wrote candidate PDB: results/stage09/validation_top3/pdbs/candidate_2.pdb
Wrote candidate PDB: results/stage09/validation_top3/pdbs/candidate_3.pdb

Top structural summary:
 stage08_structural_rank  sample_id        generation_regime  final_multimodal_rank_score  esmfold_mean_plddt  mutation_site_mean_plddt  rmsd_to_selected_seed  mutation_site_confidence_ge70_fraction  stage08_pass                                            stage08_decision_reason
                       1          1 stage09_localized_search                     0.335713            0.219675                    0.2175              13.341363

## Step 6 — Build the final Stage 09 report

In [11]:
# Apply the import json patch to the script automatically
file_path = ROOT / "scripts" / "09g_make_stage09_report.py"
text = file_path.read_text()
if "import json" not in text:
    text = text.replace("import pandas as pd", "import pandas as pd\nimport json")
    file_path.write_text(text)
    print("Patched: added import json")

cmd = [
    sys.executable, "scripts/09g_make_stage09_report.py",
    "--search_csv", f"{OUT_ROOT}/search/stage09_search_candidates.csv",
    "--prefilter_csv", f"{OUT_ROOT}/prefilter/stage09_prefilter_top12.csv",
    "--validation_csv", f"{OUT_ROOT}/validation_top10/stage08_structural_fasttrack_summary.csv",
    "--out_dir", f"{OUT_ROOT}/report",
]

if BASELINE_STAGE08 is not None and Path(BASELINE_STAGE08).exists():
    cmd += ["--baseline_stage08_csv", BASELINE_STAGE08]

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

Running: /opt/conda/bin/python scripts/09g_make_stage09_report.py --search_csv results/stage09/search/stage09_search_candidates.csv --prefilter_csv results/stage09/prefilter/stage09_prefilter_top12.csv --validation_csv results/stage09/validation_top10/stage08_structural_fasttrack_summary.csv --out_dir results/stage09/report


Wrote: results/stage09/report/stage09_final_candidate_table.csv
Wrote: results/stage09/report/stage09_summary.json
Wrote: results/stage09/report/stage09_report.md


CompletedProcess(args=['/opt/conda/bin/python', 'scripts/09g_make_stage09_report.py', '--search_csv', 'results/stage09/search/stage09_search_candidates.csv', '--prefilter_csv', 'results/stage09/prefilter/stage09_prefilter_top12.csv', '--validation_csv', 'results/stage09/validation_top10/stage08_structural_fasttrack_summary.csv', '--out_dir', 'results/stage09/report'], returncode=0)

## Step 7 — Inspect outputs

In [12]:
import pandas as pd
from IPython.display import display

display(pd.read_csv(f"{OUT_ROOT}/prefilter/stage09_prefilter_top12.csv").head(12))
display(pd.read_csv(f"{OUT_ROOT}/validation_top10/stage08_structural_fasttrack_summary.csv"))

report_path = Path(f"{OUT_ROOT}/report/stage09_report.md")
if report_path.exists():
    print(report_path.read_text())
else:
    print(f"[WARN] Report markdown not found: {report_path}")

,candidate_sequence,parent_sequence,parent_id,proposal_position,proposal_aa,proposal_rank_within_parent,round_index,mutations,target_probability,seed_cosine,...,diversity_penalty,candidate_id,sample_id,generation_regime,mutation_positions,editable_hotspots,prefilter_diversity_penalty,prefilter_rank,prefilter_reason,final_multimodal_rank_score
0,MGFYAGRIGDKKVLSLTSGNNKDVNNHTNPGWDTIFHSDMPHVVVL...,MGFYAGRIGDKKVLSLTSGNNKDVNNHTNPGWDTIFHSDMPHVVVL...,r3_cand_1,505,W,3,4,"['373:T→W', '391:L→I', '402:N→S', '505:G→W']",0.024834,0.999880,...,0.000000,r4_cand_1,1,stage09_localized_search,373:T→W;391:L→I;402:N→S;505:G→W,"291,329,334,338,349,373,391,400,401,402,425,42...",0.000000,1,kept_by_stage09_structural_prefilter,0.335713
1,MGFYAGRIGDKKVLSLTSGNNKDVNNHTNPGWDTIFHSDMPHVVVL...,MGFYAGRIGDKKVLSLTSGNNKDVNNHTNPGWDTIFHSDMPHVVVL...,r3_cand_1,427,E,2,4,"['373:T→W', '391:L→I', '402:N→S', '427:T→E']",0.022967,0.999930,...,0.999956,r4_cand_2,2,stage09_localized_search,373:T→W;391:L→I;402:N→S;427:T→E,"291,329,334,338,349,373,391,400,401,402,425,42...",0.999925,2,kept_by_stage09_structural_prefilter,0.335432
2,MGFYAGRIGDKKVLSLTSGNNKDVNNHTNPGWDTIFHSDMPHVVVL...,MGFYAGRIGDKKVLSLTSGNNKDVNNHTNPGWDTIFHSDMPHVVVL...,r3_cand_1,400,A,18,4,"['373:T→W', '391:L→I', '400:R→A', '402:N→S']",0.030751,0.999888,...,0.999964,r4_cand_3,3,stage09_localized_search,373:T→W;391:L→I;400:R→A;402:N→S,"291,329,334,338,349,373,391,400,401,402,425,42...",0.999895,3,kept_by_stage09_structural_prefilter,0.335353
3,MGFYAGRIGDKKVLSLTSGNNKDVNNHTNPGWDTIFHSDMPHVVVL...,MGFYAGRIGDKKVLSLTSGNNKDVNNHTNPGWDTIFHSDMPHVVVL...,r3_cand_1,334,K,11,4,"['334:M→K', '373:T→W', '391:L→I', '402:N→S']",0.029721,0.999882,...,0.999941,r4_cand_5,5,stage09_localized_search,334:M→K;373:T→W;391:L→I;402:N→S,"291,329,334,338,349,373,391,400,401,402,425,42...",0.999887,4,kept_by_stage09_structural_prefilter,0.335212
4,MGFYAGRIGDKKVLSLTSGNNKDVNNHTNPGWDTIFHSDMPHVVVL...,MGFYAGRIGDKKVLSLTSGNNKDVNNHTNPGWDTIFHSDMPHVVVL...,r3_cand_1,527,W,4,4,"['373:T→W', '391:L→I', '402:N→S', '527:D→W']",0.024119,0.999835,...,0.999922,r4_cand_4,4,stage09_localized_search,373:T→W;391:L→I;402:N→S;527:D→W,"291,329,334,338,349,373,391,400,401,402,425,42...",0.999892,5,kept_by_stage09_structural_prefilter,0.335212
5,MGFYAGRIGDKKVLSLTSGNNKDVNNHTNPGWDTIFHSDMPHVVVL...,MGFYAGRIGDKKVLSLTSGNNKDVNNHTNPGWDTIFHSDMPHVVVL...,r3_cand_4,400,A,18,4,"['373:T→W', '391:L→I', '400:R→A', '505:G→W']",0.031667,0.999838,...,0.999967,r4_cand_6,6,stage09_localized_search,373:T→W;391:L→I;400:R→A;505:G→W,"291,329,334,338,349,373,391,400,401,402,425,42...",0.999957,6,kept_by_stage09_structural_prefilter,0.335180
6,MGFYAGRIGDKKVLSLTSGNNKDVNNHTNPGWDTIFHSDMPHVVVL...,MGFYAGRIGDKKVLSLTSGNNKDVNNHTNPGWDTIFHSDMPHVVVL...,r3_cand_19,400,A,18,4,"['334:M→K', '373:T→W', '391:L→I', '400:R→A']",0.037384,0.999841,...,0.999967,r4_cand_7,7,stage09_localized_search,334:M→K;373:T→W;391:L→I;400:R→A,"291,329,334,338,349,373,391,400,401,402,425,42...",0.999932,7,kept_by_stage09_structural_prefilter,0.335049
7,MGFYAGRIGDKKVLSLTSGNNKDVNNHTNPGWDTIFHSDMPHVVVL...,MGFYAGRIGDKKVLSLTSGNNKDVNNHTNPGWDTIFHSDMPHVVVL...,r3_cand_1,334,A,7,4,"['334:M→A', '373:T→W', '391:L→I', '402:N→S']",0.026676,0.999920,...,0.999951,r4_cand_8,8,stage09_localized_search,334:M→A;373:T→W;391:L→I;402:N→S,"291,329,334,338,349,373,391,400,401,402,425,42...",0.999912,8,kept_by_stage09_structural_prefilter,0.335029
8,MGFYAGRIGDKKVLSLTSGNNKDVNNHTNPGWDTIFHSDMPHVVVL...,MGFYAGRIGDKKVLSLTSGNNKDVNNHTNPGWDTIFHSDMPHVVVL...,r3_cand_2,400,A,18,4,"['373:T→W', '400:R→A', '402:N→S', '505:G→W']",0.030984,0.999815,...,0.999980,r4_cand_9,9,stage09_localized_search,373:T→W;400:R→A;402:N→S;505:G→W,"291,329,334,338,349,373,391,400,401,402,425,42...",0.999947,9,kept_by_stage09_structural_prefilter,0.335027
9,MGFYAGRIGDKKVLSLTSGNNKDVNNHTNPGWDTIFHSDMPHVVVL...,MGFYAGRIGDKKVLSLTSGNNKDVNNHTNPGWDTIFHSDMPHVVVL...,r3_cand_2,527,W,4,4,"['373:T→W', '402:N→S', '505:G→W', '527:D→W']",0.024655,0.999746,...,0.999956,r4_cand_10,10,stage09_localized_search,373:T→W;

,parent_sequence,parent_id,proposal_position,proposal_aa,proposal_rank_within_parent,round_index,mutations,target_probability,seed_cosine,family_cosine,...,esmfold_ptm,rmsd_to_selected_seed,mutation_site_confidence_ge70_fraction,mutation_site_mean_plddt,seed_esmfold_mean_plddt,seed_esmfold_ptm,candidate_pdb,stage08_structural_rank,stage08_pass,stage08_decision_reason
0,MGFYAGRIGDKKVLSLTSGNNKDVNNHTNPGWDTIFHSDMPHVVVL...,r3_cand_1,505,W,3,4,"['373:T→W', '391:L→I', '402:N→S', '505:G→W']",0.024834,0.999880,0.998892,...,0.154666,13.341363,0.0,0.2175,0.224767,0.147549,results/stage09/validation_top10/pdbs/candidat...,1,False,low_global_confidence;high_seed_drift;low_muta...
1,MGFYAGRIGDKKVLSLTSGNNKDVNNHTNPGWDTIFHSDMPHVVVL...,r3_cand_1,427,E,2,4,"['373:T→W', '391:L→I', '402:N→S', '427:T→E']",0.022967,0.999930,0.998760,...,0.153640,14.073246,0.0,0.2325,0.224767,0.147549,results/stage09/validation_top10/pdbs/candidat...,2,False,low_global_confidence;high_seed_drift;low_muta...
2,MGFYAGRIGDKKVLSLTSGNNKDVNNHTNPGWDTIFHSDMPHVVVL...,r3_cand_1,400,A,18,4,"['373:T→W', '391:L→I', '400:R→A', '402:N→S']",0.030751,0.999888,0.998865,...,0.151293,13.129426,0.0,0.2175,0.224767,0.147549,results/stage09/validation_top10/pdbs/candidat...,3,False,low_global_confidence;high_seed_drift;low_muta...
3,MGFYAGRIGDKKVLSLTSGNNKDVNNHTNPGWDTIFHSDMPHVVVL...,r3_cand_1,527,W,4,4,"['373:T→W', '391:L→I', '402:N→S', '527:D→W']",0.024119,0.999835,0.998781,...,0.154199,11.982859,0.0,0.2250,0.224767,0.147549,results/stage09/validation_top10/pdbs/candidat...,4,False,low_global_confidence;high_seed_drift;low_muta...
4,MGFYAGRIGDKKVLSLTSGNNKDVNNHTNPGWDTIFHSDMPHVVVL...,r3_cand_1,334,K,11,4,"['334:M→K', '373:T→W', '391:L→I', '402:N→S']",0.029721,0.999882,0.998900,...,0.151247,12.536547,0.0,0.2175,0.224767,0.147549,results/stage09/validation_top10/pdbs/candidat...,5,False,low_global_confidence;high_seed_drift;low_muta...
5,MGFYAGRIGDKKVLSLTSGNNKDVNNHTNPGWDTIFHSDMPHVVVL...,r3_cand_4,400,A,18,4,"['373:T→W', '391:L→I', '400:R→A', '505:G→W']",0.031667,0.999838,0.998968,...,0.152450,14.086135,0.0,0.2275,0.224767,0.147549,results/stage09/validation_top10/pdbs/candidat...,6,False,low_global_confidence;high_seed_drift;low_muta...
6,MGFYAGRIGDKKVLSLTSGNNKDVNNHTNPGWDTIFHSDMPHVVVL...,r3_cand_19,400,A,18,4,"['334:M→K', '373:T→W', '391:L→I', '400:R→A']",0.037384,0.999841,0.998958,...,0.150529,12.994794,0.0,0.2250,0.224767,0.147549,results/stage09/validation_top10/pdbs/candidat...,7,False,low_global_confidence;high_seed_drift;low_muta...
7,MGFYAGRIGDKKVLSLTSGNNKDVNNHTNPGWDTIFHSDMPHVVVL...,r3_cand_1,334,A,7,4,"['334:M→A', '373:T→W', '391:L→I', '402:N→S']",0.026676,0.999920,0.998604,...,0.152240,11.890279,0.0,0.2200,0.224767,0.147549,results/stage09/validation_top10/pdbs/candidat...,8,False,low_global_confidence;high_seed_drift;low_muta...
8,MGFYAGRIGDKKVLSLTSGNNKDVNNHTNPGWDTIFHSDMPHVVVL...,r3_cand_2,400,A,18,4,"['373:T→W', '400:R→A', '402:N→S', '505:G→W']",0.030984,0.999815,0.998978,...,0.150891,15.125929,0.0,0.2175,0.224767,0.147549,results/stage09/validation_top10/pdbs/candidat...,9,False,low_global_confidence;high_seed_drift;low_muta...
9,MGFYAGRIGDKKVLSLTSGNNKDVNNHTNPGWDTIFHSDMPHVVVL...,r3_cand_2,527,W,4,4,"['373:T→W', '402:N→S', '505:G→W', '527:D→W']",0.024655,0.999746,0.998857,...,0.152668,12.273642,0.0,0.2325,0.224767,0.147549,results/stage09/validation_top10/pdbs/candidat...,10,False,low_global_confidence;high_seed_drift;low_muta...


# Stage 09 structure-aware redesign report

## Stage 09 summary

- search candidates generated: **90**
- prefilter survivors: **12**
- structurally validated candidates: **10**
- structural pass count: **0**
- structural pass rate: **0.000**
- mean ESMFold pLDDT: **0.224**
- mean mutation-site pLDDT: **0.223**
- mean RMSD to seed: **13.143 Å**

## Top validated candidates

| rank | sample_id | stage09_score | mean_pLDDT | mut_mean_pLDDT | RMSD_to_seed | pass | reason |
|---:|---:|---:|---:|---:|---:|:---:|---|
| 1 | 3 | 0.335353 | 0.23 | 0.22 | 13.129 | False | low_global_confidence;high_seed_drift;low_mutation_site_confidence |
| 2 | 2 | 0.335432 | 0.23 | 0.23 | 14.073 | False | low_global_confidence;high_seed_drift;low_mutation_site_confidence |
| 3 | 10 | 0.335015 | 0.23 | 0.23 | 12.274 | False | low_global_confidence;high_seed_drift;low_mutation_site_confidence |
| 4 | 4 | 0.335212 | 0.22 | 0.22 | 11.983 | False | low_global_confidence;high_seed_drift;low_mutation_site_confidence |

## Step 8 — Export Backup Bundle

In [13]:
!cd /home/sagemaker-user/phageforge_clean && tar -czf stage09_FINAL_bundle.tar.gz \
    phageforge \
    scripts \
    stage07 \
    results \
    seed_42 \
    rbp_dataset_eskapee_strict.csv \
    pyproject.toml \

!ls -lh /home/sagemaker-user/phageforge_clean | grep FINAL

-rw-r--r--.  1 sagemaker-user users 2.2M May 18 18:13 stage09_FINAL_bundle.tar.gz
